# NFL 4th Down Decision Analysis
## Data Collection & Processing

This notebook pulls play-by-play data for all NFL seasons from 2016–2025 using the `nfl_data_py` library, filters to 4th down plays, and saves a clean processed dataset for modeling and analysis.

---

### Setup
Configure file paths relative to the project root so the notebook works regardless of where it is run from.

In [1]:
import nfl_data_py as nfl
import os

# Set path relative to project root, not notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_path = os.path.join(project_root, "data", "raw")
processed_path = os.path.join(project_root, "data", "processed")

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")

Raw data path: /workspaces/nfl-4th-down-analysis/data/raw
Processed data path: /workspaces/nfl-4th-down-analysis/data/processed


Paths confirmed. Raw data will be stored in `data/raw/` as individual parquet files per season. Processed data will be saved to `data/processed/` after filtering and feature engineering.

---

### Download Play-by-Play Data
Pull one season at a time to avoid memory issues. Each season is saved immediately to disk and cleared from memory before the next pull.

In [2]:
seasons = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

for season in seasons:
    print(f"Pulling {season}...")
    df = nfl.import_pbp_data([season])
    df.to_parquet(os.path.join(raw_path, f"pbp_{season}.parquet"), index=False)
    print(f"{season} saved.")
    del df

print("All seasons saved.")

Pulling 2016...
2016 done.
Downcasting floats.
2016 saved.
Pulling 2017...
2017 done.
Downcasting floats.
2017 saved.
Pulling 2018...
2018 done.
Downcasting floats.
2018 saved.
Pulling 2019...
2019 done.
Downcasting floats.
2019 saved.
Pulling 2020...
2020 done.
Downcasting floats.
2020 saved.
Pulling 2021...
2021 done.
Downcasting floats.
2021 saved.
Pulling 2022...
2022 done.
Downcasting floats.
2022 saved.
Pulling 2023...
2023 done.
Downcasting floats.
2023 saved.
Pulling 2024...
2024 done.
Downcasting floats.
2024 saved.
Pulling 2025...
2025 done.
Downcasting floats.
2025 saved.
All seasons saved.


All 10 seasons successfully downloaded and saved as parquet files. Each file contains ~49,000 plays and 396 columns covering every recorded play from the 2016 through 2025 NFL seasons.

Parquet format was chosen over CSV for three reasons:
- **Speed** — significantly faster read/write than CSV
- **Size** — compressed automatically, much smaller on disk
- **Type preservation** — column data types are retained on load, no guessing required

---

### Verify Downloads
Confirm all 10 files exist on disk and that a sample season loads correctly before proceeding.

In [3]:
import pandas as pd

files = os.listdir(raw_path)
print(f"Files in data/raw: {sorted(files)}")

# Load one season to confirm it reads correctly
test = pd.read_parquet(os.path.join(raw_path, "pbp_2023.parquet"))
print(f"\n2023 shape: {test.shape}")
print(f"Columns: {test.columns.tolist()[:10]}...")
del test

Files in data/raw: ['pbp_2016.parquet', 'pbp_2017.parquet', 'pbp_2018.parquet', 'pbp_2019.parquet', 'pbp_2020.parquet', 'pbp_2021.parquet', 'pbp_2022.parquet', 'pbp_2023.parquet', 'pbp_2024.parquet', 'pbp_2025.parquet']

2023 shape: (49665, 396)
Columns: ['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam']...


All 10 season files confirmed on disk. The 2023 season contains 49,665 plays across 396 columns, consistent with expectations. The dataset includes every recorded play type — passes, runs, punts, field goals, kickoffs, penalties, and more.

With 396 columns available we don't need everything — the next step selects only the columns relevant to 4th down decision modeling.

---

### Filter to 4th Down Plays
Select the 27 columns needed for analysis, filter to 4th down plays only, and remove non-decision plays like penalties (`no_play`). Then engineer two key columns:
- `coach` — identifies which coach made the decision on each play
- `decision` — simplifies play type into `go`, `punt`, or `field_goal`

In [4]:
cols = [
    'play_id', 'game_id', 'season', 'week',
    'posteam', 'defteam', 'home_team', 'away_team',
    'home_coach', 'away_coach', 'posteam_type',
    'down', 'ydstogo', 'yardline_100',
    'score_differential', 'game_seconds_remaining',
    'qtr', 'goal_to_go',
    'play_type', 'fourth_down_converted', 'fourth_down_failed',
    'epa', 'wp', 'wpa',
    'field_goal_result', 'punt_attempt', 'field_goal_attempt',
    'offense_formation', 'offense_personnel',
    'defenders_in_box', 'defense_personnel',
    'defense_man_zone_type', 'defense_coverage_type'
]

dfs = []
for season in range(2016, 2026):
    df = pd.read_parquet(os.path.join(raw_path, f"pbp_{season}.parquet"), columns=cols)
    df = df[df['down'] == 4]
    df = df[df['play_type'].isin(['pass', 'run', 'punt', 'field_goal'])]
    dfs.append(df)
    print(f"{season}: {len(df)} 4th down plays")

fourth_downs = pd.concat(dfs, ignore_index=True)

fourth_downs['coach'] = fourth_downs.apply(
    lambda row: row['home_coach'] if row['posteam_type'] == 'home' else row['away_coach'],
    axis=1
)
fourth_downs['decision'] = fourth_downs['play_type'].map({
    'pass': 'go',
    'run': 'go',
    'punt': 'punt',
    'field_goal': 'field_goal'
})

fourth_downs.to_parquet(os.path.join(processed_path, "fourth_downs.parquet"), index=False)
print(f"\nTotal shape: {fourth_downs.shape}")
print(f"\nDecision breakdown:\n{fourth_downs['decision'].value_counts()}")

2016: 3891 4th down plays
2017: 4027 4th down plays
2018: 3782 4th down plays
2019: 3801 4th down plays
2020: 3596 4th down plays
2021: 3982 4th down plays
2022: 4071 4th down plays
2023: 4222 4th down plays
2024: 3998 4th down plays
2025: 3997 4th down plays

Total shape: (39367, 35)

Decision breakdown:
decision
punt          22519
field_goal     9790
go             7058
Name: count, dtype: int64


The processed dataset contains **39,367 4th down plays** across 10 seasons with 35 columns — down from 396 in the raw data. This version adds 6 formation and personnel columns to the original 29, enabling deeper analysis of play design and defensive context.

**Decision breakdown:**
| Decision | Count | Rate |
|---|---|---|
| Punt | 22,519 | 57% |
| Field Goal | 9,790 | 25% |
| Go For It | 7,058 | 18% |

Teams went for it on only **18% of 4th downs** across the entire dataset. This conservative baseline is the inefficiency our model will quantify — analytically, teams should be going for it far more often in many of these situations.

`no_play` penalty plays were excluded since they don't represent an actual coaching decision. The `coach` column was derived by matching the possession team (`posteam_type`) to either `home_coach` or `away_coach`, ensuring we correctly attribute each decision to the right sideline.

**Formation and personnel columns added:**
- `offense_formation` — offensive formation at the snap
- `offense_personnel` — offensive personnel grouping
- `defenders_in_box` — number of defenders in the box
- `defense_personnel` — defensive personnel grouping
- `defense_man_zone_type` — man vs zone coverage
- `defense_coverage_type` — specific coverage type

---

### Next Steps
With the processed dataset saved to `data/processed/fourth_downs.parquet`, we move to exploratory data analysis in `02_eda.ipynb` to visualize trends in coach aggressiveness before building the predictive model.